In [ ]:
%load_ext autoreload
%autoreload 2
import os
import sys
import numpy as np
import pandas as pd
import xarray as xr
import umap
from scipy.ndimage import gaussian_filter
import plotly.express as px
from os.path import join as pjoin
from tqdm.notebook import tqdm
import plotly.graph_objects as go
from scipy.stats import pearsonr, spearmanr, zscore

sys.path.append('../')
import circletrack_behavior as ctb
import circletrack_neural as ctn
import plotting_functions as pf
import place_cells as pc
import pca_ica as pca

xr.set_options(keep_attrs=True)

In [ ]:
## Settings
project_folder = ['MultiCon_Imaging']
experiment_folders = ['MultiCon_Imaging5', 'MultiCon_Imaging6', 'MultiCon_Imaging7']
dpath = f'../../{project_folder[0]}'
fig_path = f'../../../Manuscripts/MultiCon/intermediate_plots/UMAP'
int_data = f'../../../Manuscripts/MultiCon/intermediate_plots/intermediate_data'
chance_color = '#7d7d7d'
avg_color = '#287347'
subject_color = '#7d7d7d'
ce_colors = ['#7A22BC', '#378616']
ce_colors_dict = {'Two-context': '#378616', 'Multi-context': '#7A22BC'}
symbol_dict = {'Two-context': 'x', 'Multi-context': 'circle'}
symbols_list = ['x', 'circle']
context_colors = {'A': '#a9a9a9', 'B': '#dc267f', 'C': '#648fff', 'D': '#fe6100',
                  'A1-5': '#a9a9a9', 'A5-10': '#dc267f', 'A10-15': '#648fff'}
mouse_colors = ['midnightblue', 'darkred', 'darkorchid', 'darkturquoise']
session_list = [f'A{x}' for x in np.arange(1, 6)] + [f'B{x}' for x in np.arange(1, 6)] + [f'C{x}' for x in np.arange(1, 6)] + [f'D{x}' for x in np.arange(1, 6)]
control_list = [f'A{x}' for x in np.arange(1, 16)] + [f'B{x}' for x in np.arange(1, 6)]
day_list = [f'Day {x}' for x in np.arange(1, 21)]
bin_size = 0.06 ## 0.06 radians linear position equivalent to 2cm-wide bins
reward_bin_size = 0.09
all_midpoints = np.arange(-(reward_bin_size * 3), (reward_bin_size * 3) + reward_bin_size, reward_bin_size)
time_bin_size = 0.2 ## in seconds
velocity_thresh = 10
centroid_distance = 4
data_of_interest = 'place_cells' ## one of behav, aligned_minian, aligned_place_cells, lin_behav
data_type = 'S'
conversion = 2 / 0.06 ## 2cm per 0.06 radians

if not os.path.exists(fig_path):
    os.makedirs(fig_path)

xr.set_options(keep_attrs=True)

### Single mouse example - plot the activity from one day in one context after unsupervised UMAP dimensionality reduction.

In [ ]:
## Settings
experiment = 'MultiCon_Imaging5'
mouse = 'mc51'
day_of_int = '16'
session = f'{mouse}_{data_type}_{day_of_int}.nc'
only_running = True
correct_dir = True
cell_type = 'all_cells'
bin_size_seconds = 0.1 ## 100ms bins
fps = 30 ## recorded at 30 frames per second
normalize = False
bin_data = False

In [ ]:
## Load data
exp_path = pjoin(dpath, f'{experiment}/output/{data_of_interest}/')
mpath = pjoin(exp_path, f'{mouse}/{data_type}')

sdata = xr.open_dataset(pjoin(mpath, session))[data_type] ## don't need to perform ctn.qc_matrix because aligned_place_cells already was qced
## Subset data for correct direction and when running
neural_data, _ = ctn.subset_correct_dir_and_running(sdata, 
                                                    correct_dir=correct_dir, 
                                                    only_running=only_running,
                                                    velocity_thresh=velocity_thresh)
## Bin activity to help reduce noise
if bin_data:
    act = ctn.bin_activity(neural_data.values, 
                            bin_size_seconds=bin_size_seconds,
                            fps=fps,
                            func=np.mean)
else:
    act = neural_data.values
## If normalize = True, zscore data. May be overkill - looking at the distribution of max deconvolved event amplitudes, there is a small right tail.
## Distribution is between 0 and 3.7
if normalize:
    actz = zscore(act, axis=1)
else:
    actz = act.copy()
## Unsupervised UMAP
embedding = umap.UMAP(n_neighbors=40,
                      n_components=3,
                      min_dist=0.6, 
                      learning_rate=1.0,
                      metric='cosine').fit_transform(actz.T)

In [ ]:
lick_emb = embedding[neural_data['lick_port'] != -1, :]
rw_emb = embedding[neural_data['water'], :]
fig = pf.custom_graph_template(x_title='', y_title='')
fig.add_trace(go.Scatter3d(x=embedding[:, 0], y=embedding[:, 1], z=embedding[:, 2], mode='markers', marker_size=2, marker_color='darkgrey'))
fig.add_trace(go.Scatter3d(x=lick_emb[:, 0], y=lick_emb[:, 1], z=lick_emb[:, 2], mode='markers', marker_size=2, marker_color='red'))
fig.add_trace(go.Scatter3d(x=rw_emb[:, 0], y=rw_emb[:, 1], z=rw_emb[:, 2], mode='markers', marker_size=2, marker_color='darkorchid'))
fig.show()
# fig.write_image(pjoin(fig_path, f'{mouse}_{day_of_int}_unsupervised_umap.png'))

### Single mouse example - plot the activity from one day in one context after unsupervised UMAP dimensionality reduction after binning into angle bins.

In [ ]:
normalize = False
only_running = False
correct_dir = False
bin_size_deg = 10 ## in degrees

## Load data
exp_path = pjoin(dpath, f'{experiment}/output/{data_of_interest}/')
mpath = pjoin(exp_path, f'{mouse}/{data_type}')

sdata = xr.open_dataset(pjoin(mpath, session))[data_type] ## don't need to perform ctn.qc_matrix because aligned_place_cells already was qced
## Subset data for correct direction and when running
neural_data, _ = ctn.subset_correct_dir_and_running(sdata, 
                                                    correct_dir=correct_dir, 
                                                    only_running=only_running,
                                                    velocity_thresh=velocity_thresh)
angles = np.arange(0, 360, bin_size_deg)
angle_ar = np.zeros((neural_data.shape[0], angles.shape[0])) ## neuron by angle
for idx, angle in enumerate(angles):
    loop_data = neural_data[:, (neural_data['a_pos'] >= angle) & (neural_data['a_pos'] < angle + bin_size_deg)]
    angle_ar[:, idx] = loop_data.mean(dim='frame')
if normalize:
    actz = zscore(angle_ar, axis=1)
else:
    actz = angle_ar.copy()
## Unsupervised UMAP
embedding = umap.UMAP(n_neighbors=40,
                      n_components=2,
                      min_dist=0.6, 
                      learning_rate=1.0,
                      metric='cosine').fit_transform(actz.T)

In [ ]:
## 3D
fig = pf.custom_graph_template(x_title='', y_title='')
fig.add_trace(go.Scatter3d(x=embedding[:, 0], y=embedding[:, 1], z=embedding[:, 2], mode='lines'))
fig.show()
fig.write_image(pjoin(fig_path, f'{mouse}_{day_of_int}_{bin_size_deg}angle_bin_unsupervised_umap.png'))

In [ ]:
## 2D
fig = pf.custom_graph_template(x_title='', y_title='')
fig.add_trace(go.Scattergl(x=embedding[:, 0], y=embedding[:, 1], mode='lines+markers'))
fig.show()
# fig.write_image(pjoin(fig_path, f'{mouse}_{day_of_int}_{bin_size_deg}angle_bin_unsupervised_umap.png'))

### One mouse - supervised trial blocks.

In [ ]:
number_of_binned_trials = 1
only_running = True 
correct_dir = True
bin_size_seconds = 0.1 ## 100ms bins
fps = 30 ## recorded at 30 frames per second
normalize = False
bin_data = True
samples_per_bin = bin_size_seconds * fps

## Load data
sdata = xr.open_dataset(pjoin(dpath, f'{mouse}/{data_type}/{mouse}_{data_type}_{day}.nc'))[data_type]
## Label trial block
trial_block = np.zeros(sdata['frame'].shape[0])
trial_data = sdata.assign_coords(trial_block = ('frame', trial_block))

block_num = 1
for trial in np.unique(trial_data['trials'] + 1): ## plus 1 to avoid 0 % 5, since trials start at 0
    if trial % number_of_binned_trials != 0:
        block_num = block_num
        trial_data['trial_block'][trial_data['trials'] + 1 == trial] = block_num
    else:
        trial_data['trial_block'][trial_data['trials'] + 1 == trial] = block_num
        block_num += 1
## Subset data for correct direction and when running
neural_data, _ = ctn.subset_correct_dir_and_running(trial_data, 
                                                    correct_dir=correct_dir, 
                                                    only_running=only_running,
                                                    velocity_thresh=velocity_thresh)
## Bin data
if bin_data:
    act = ctn.bin_activity(neural_data.values, 
                           bin_size_seconds=bin_size_seconds,
                           fps=fps,
                           func=np.mean)
    binned_trial_blocks = neural_data['trial_block'][::int(samples_per_bin)].values
else:
    act = neural_data.values
    binned_trial_blocks = neural_data['trial_block']
## If normalize = True, zscore data. May be overkill - looking at the distribution of max deconvolved event amplitudes, there is a small right tail.
## Distribution is between 0 and 3.7
if normalize:
    actz = zscore(act, axis=1)
else:
    actz = act.copy()
## Unsupervised UMAP
embedding = umap.UMAP(n_neighbors=40,
                      n_components=3,
                      min_dist=0.6, 
                      learning_rate=1.0,
                      metric='cosine').fit_transform(actz.T, y=binned_trial_blocks)

In [ ]:
fig = pf.custom_graph_template(x_title='', y_title='')
fig.add_trace(go.Scatter3d(x=embedding[:, 0], y=embedding[:, 1], z=embedding[:, 2], mode='lines'))
fig.show()
fig.write_image(pjoin(fig_path, f'{mouse}_{day}_supervised_trial_blocks_umap.png'))

In [ ]:
fig = pf.custom_graph_template(x_title='', y_title='', font_size=18)
prev_trial_len = 0
for trial in np.unique(binned_trial_blocks):
    trial_len = actz[:, binned_trial_blocks == trial].shape[1]
    plot_data = embedding[prev_trial_len:prev_trial_len+trial_len, :]
    prev_trial_len += trial_len

    fig.add_trace(go.Scatter3d(x=plot_data[:, 0], y=plot_data[:, 1], z=plot_data[:, 2], mode='lines', name=f'Block {trial}', showlegend=True))
fig.show()
# fig.write_image(pjoin(fig_path, f'{mouse}_{day}_supervised_colored_by_trial_block.png'))